# Project 3 - NCSU ST 554
#### Author: Trevor Lillywhite
#### Due Date: April 30, 2026

## Introduction

**Purpose:** This project will focus on using `spark` to handle streaming data, fitting a machine learning model, and performing other associated tasks. 

It will involve both this notebook (using a pySpark3 kernel) and producing a `.py` file to read in a stream of data from a static source. After a portion of the dataset is used for model training, the remaining portions will be incrementally streamed in the form of csv files to a folder being monitored by the code. The machine learning model will do prediction on this stream and write the results out to the console.

**Data Source:** The data is adapted from a UCI machine learning dataset on a study predicting power consumption to factors like time of day, temperature, and humidity. More information on the original dataset is available here: 
+ UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city
+ Original Case Study: https://www.semanticscholar.org/paper/Comparison-of-Machine-Learning-Algorithms-for-the-%3A-Salam-Hibaoui/177512e766fe5d8624a1b3e93abd11082ac37b3f


## Fitting the Model

### Reading in Data

The data is saved as a csv in the same directory level as this notebook. However, it can also be accessed at the following URL: 
+ https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv

We will read in the data as a standard `pandas` DataFrame using the `pd.read_csv()` function.

In [4]:
# Import relevant modules
import pandas as pd
import numpy as np
import time
from pyspark.sql.types import StructType
from pyspark.sql import SparkSession

# Create spark session
spark = SparkSession.builder.getOrCreate()

In [5]:
# Read in data to pandas DataFrame
df_pandas = pd.read_csv('power_ml_data.csv', sep=',', header=0)

# Verify data
df_pandas.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [6]:
# Check dataframe info
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47174 entries, 0 to 47173
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Temperature            47174 non-null  float64
 1   Humidity               47174 non-null  float64
 2   Wind_Speed             47174 non-null  float64
 3   General_Diffuse_Flows  47174 non-null  float64
 4   Diffuse_Flows          47174 non-null  float64
 5   Power_Zone_1           47174 non-null  float64
 6   Power_Zone_2           47174 non-null  float64
 7   Power_Zone_3           47174 non-null  float64
 8   Month                  47174 non-null  int64  
 9   Hour                   47174 non-null  int64  
dtypes: float64(8), int64(2)
memory usage: 3.6 MB


It appears that the data was read in properly. All columns are numerical (float64 or int64) with no missing values (47174 non-null values for each column). 

Now we will convert this to a `spark` DataFrame.

In [8]:
# Convert to spark DataFrame
df = spark.createDataFrame(df_pandas)
df.show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

The data appears consistent after the format change.  


In [ ]:
### Data Preparation

